In [ ]:
### 

### TODO REMOVE AUTOMATED COMMENTS
# Tweak clustering parameters for topics
# Run run run run run

In [2]:
from bertopic import BERTopic
import sqlite3
import re
import pandas as pd
from datetime import datetime, UTC

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from datasets import Dataset

from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from typing import Dict, List, Tuple
import unicodedata
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance
from umap import UMAP
from bertopic.vectorizers import ClassTfidfTransformer

In [3]:
conn1 = sqlite3.connect("starrail_weekly.db")

In [ ]:
#comment_data = pd.read_sql_query("SELECT comment_id, post_id, comment_body FROM comments",conn1)

In [5]:
### Cleaning comments
# Removing NaN, <image>, "" (empty string)
# Removing automated comments (post removed for rule breaking, spoilers)
# Comments must have at least 4 words, 3 unique, and average word length > 1.5 (for funny stuff e.g. R E A L, P E A K)

# Filter words for bot comments
AUTOMATED_COMMENTS = [
    r"please keep in mind our spoiler policy",
    r"removed for breaking Rule",
    r"Hey Trailblazer! We see that you posted this with the",
    r"^\[removed]",
    r"^\[deleted]",
    r"Hey Trailblazer! We see that you posted this with the",
    r"Unfortunately, your content had to be removed",
    r"Processing img ",
    r"source",
    r"Shipping Rules",
    r"Thank you for sharing",
    r"Unfortunately, your comment was removed due to",
    r"Unfortunately, your content was removed due to",
    r"Thanks for submitting a Tech Help question",
    r"^\[ Removed by Reddit ]",
    r"^\[Removed by Reddit]",
    r"View link Info",
    r"Other Threads: Weekly Gacha and Drops Megathread",
    r"Honkai: Star Rail's Third Anniversary Grand Opening!",
    r"01001111",
    r"01101111",
    r"View link\nInfo",
    r"\nWeekly Gacha and Drops Megathread",
    r"One day,? after dinner",
    r"Sauce is on my\nInstagram\nand\nTwitter\n!",
    r"FOR REFERENCE:\nPlease be noted that these definitions",
    r"🔄📅👶🦢➕📅⬅️🥱📥👨🦫🪵🌱",
    r"--- -. .",
    
    #View link Info | Feedback | Donate | DMCA | reddit video downloader | twitter video downloader

#Log in and participate in the web event to win a share
]

automated_regex = "|".join(AUTOMATED_COMMENTS)

def clean_comment_data(df: pd.DataFrame)->pd.DataFrame:
    return (df.dropna(subset=["comment_body","comment_id"])
        .loc[
            lambda df:(df["comment_body"]!="")
            & (df["comment_body"].str.split().str.len()>=4)
            & (df["comment_body"].str.lower().str.split().apply(lambda words: len(set(words)))>=3)
            & (df["comment_body"].str.split().apply(lambda words: sum(len(w) for w in words)/len(words) if words else 0)>1.5)
            & ~df["comment_body"].str.contains("<image>")
            & ~df["comment_id"].str.startswith("t3")
            & ~df["comment_body"].str.contains(automated_regex,case=False,na=False)
            & ~df["comment_body"].str.match(r"^\s*d+[\d\s]*$")
            
        ]
    .assign(text_length=lambda df: df["comment_body"].str.len())
    .sort_values("text_length")
    .reset_index(drop=True)
    )

comment_data=clean_comment_data(comment_data)

In [6]:
### Replace links and mentions (@) with "http", "@"

URL_PATTERN = re.compile(r"https?://\S+|www\.\S+")
USER_PATTERN = re.compile(r"@\w+")

def preprocess_comment(text: str, apply_jargon_map: bool = True) -> str:
    """
    Preprocessor for comments.
    Normalizes URLs/@users and maps gacha slang.
    """
    if not isinstance(text, str) or not text.strip():
        return ""

    # Normalize links and mentions
    text = URL_PATTERN.sub("http", text)
    text = USER_PATTERN.sub("@user", text)

    return " ".join(text.split())

# Preprocess
comment_data["comment_body"]=comment_data["comment_body"].apply(preprocess_comment).copy()


In [7]:
### PLAYABLE CHARACTER MAP
RAW_CHARACTERS = {
    ### Standard
    "Himeko": ["himeko"],
    "Welt": ["welt"],
    "Gepard": ["gepard"],
    "Clara": ["clara"],
    "Yanqing": ["yanqing"],
    "Bronya": ["bronya"],
    "Bailu": ["bailu"],

    "Trailblazer": ["trailblazer", "tb", "hmc", "rmc", "emc", "pmc", "dmc","stelle","caelus"],

    ### 1.X
    "Seele": ["seele"],
    "Jing Yuan": ["jing yuan", "jy","king yuan"],
    "Silver Wolf": ["silver wolf", "sw","silverwolf","wolfie","wolf"],
    "Luocha": ["luocha"],
    "Blade": ["blade"],
    "Kafka": ["kafka"],
    "Dan Heng Imbibitor Lunae": ["dhil", "imbibitor lunae", "dan heng il"],
    "Fu Xuan": ["fu xuan", "fx"],
    "Jingliu": ["jingliu", "jl"],
    "Topaz & Numby": ["topaz & numby", "topaz"],
    "Huohuo": ["huohuo","huo","hou hou","hou"],
    "Argenti":["argenti"],
    "Dr. Ratio": ["dr. ratio", "dr ratio", "ratio"],
    "Ruan Mei": ["ruan mei", "rm"],

    "Dan Heng": ["dan heng", "dh"],
    "March 7th": ["march 7th", "march", "m7"],
    "Arlan": ["arlan"],
    "Asta": ["asta"],
    "Serval": ["serval"],
    "Pela": ["pela"],
    "Luka": ["luka"],
    "Hook": ["hook"],
    "Natasha": ["natasha", "nat"],
    "Sampo": ["sampo", "sampo koski"],
    "Lynx": ["lynx"],
    "Tingyun": ["tingyun", "ting"],
    "Sushang": ["sushang"],
    "Yukong": ["yukong"],
    "Hanya": ["hanya"],
    "Xueyi": ["xueyi"],
    "Guinaifen": ["guinaifen", "gwen"],
    "Qingque":["qingque","qq"],
    

    ### 2.X
    "Black Swan": ["black swan", "swan"],
    "Sparkle": ["sparkle"],
    "Acheron": ["acheron"],
    "Aventurine": ["aventurine", "aven"],
    "Boothill": ["boothill"],
    "Robin": ["robin"],
    "Firefly": ["firefly", "ff","fireflop"],
    "Jade": ["jade"],
    "Feixiao": ["feixiao"],
    "Jiaoqiu": ["jiaoqiu","jq"],
    "Yunli": ["yunli"],
    "Lingsha": ["lingsha"],
    "Rappa": ["rappa"],
    "Sunday": ["sunday"],
    "Fugue": ["fugue"],
    
    "Moze": ["moze"],
    "Gallagher":["gallagher"],
    "Misha":["misha"],

    ### 3.X
    "Herta": ["herta", "mini herta", "the herta", "therta"],
    "Aglaea": ["aglaea", "aggy"],
    "Tribbie": ["tribbie"],
    "Mydei": ["mydei"],
    "Castorice": ["castorice"],
    "Anaxa": ["anaxa"],
    "Cipher": ["cipher"],
    "Hyacine": ["hyacine","hya"],
    "Phainon": ["phainon"],
    "Archer":["archer"],
    "Saber":["saber"],
    "Cerydra":["cery","cerydra"],
    "Hysilens":["hysilens"],
    "Evernight":["evernight","e9"],
    "Dan Heng Permansor Terrae": ["dhpt", "permansor terrae", "dan heng permansor terrae", "dan heng pt"],
    "Cyrene":["cyrene",'rene'],
    "The Dahlia": ["the dahlia", "dahlia"],

    ### 4.X
    "Sparxie":["sparxie"],
    "Yaoguang":["yaoguang","yao","guang"],
    "Ashveil": ["ashveil", "ash"],
    "Silver Wolf LVL999": ["sw999"],
    "Evanescia": ["evanescia", "evan", "eva"],
    "Mortenax Blade": ["mort", "mortblade", "mblade"],
    "Himeko Nova": ["himenova"],
    "Rin Tohsaka": ["rin tohsaka", "rin"],
    "Gilgamesh": ["gilgamesh", "gil"],
    "Robin Summeretto": ["srobin", "robinetto"],
    "Aventurine Waveflair":["waven","avensp"],
    "Pearl":["pearl"],
    ("Ashveil", "Mortenax Blade"): ["ashblade", "ash blade"],
}

In [8]:
### NPC MAP
RAW_NPCS = {
    "Pompom": ["pompom", "pom-pom", "pom"],
    "Screwllum": ["screwllum","screw","screwy"],
    "Lygus": ["lygus","zandar"],
    "Pompom": ["pompom"],
    "Skott": ["scott","skott"],
    "Asat Pramad": ["asat","asat pramad","pramad"],
    "Ryusuke": ["ryusuke"],
    "Mr Reca": ["reca","mr reca","mr. reca"],
    "Sakura": ["sakura"],
    "Shirou": ["shirou"],
    "Illya": ["illya"],
    "Lancer": ["lancer"],
    "Svarog": ["svarog","mr svarog","mr. svarog"],
    "Lord Ravager": ["lord ravager","ravager"],
    "Ishtar": ["ishtar"],
    "Enkidu": ["enkidu"],
    "Motoko": ["mokoto","motoko"],
}

In [9]:
### AEON MAP
RAW_AEONS = {
    "Aha": ["aha"],
    "IX": ["ix"],
    "Nanook": ["nanook"],
    "Yaoshi": ["yaoshi"],
    "Lan": ["lan"],
    "Qlipoth": ["qlipoth"],
    "Nous": ["nous"],
    "Xipe": ["xipe"],
    "Fuli": ["fuli"],
    "Tazzyronth": ["tazzyronth"],
    "Akivili": ["akivili"],
    "Ena": ["ena"],
    "HOOH": ["hooh"],
    "Mythus": ["mythus"],
    "Terminus": ["terminus"],
}


In [10]:
def normalize_unicode(text: str) -> str:
    # Converts fancy stylized fonts (e.g. ℌ𝔢𝔯𝔱𝔞) to standard Latin characters
    return unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode('utf-8')

# Apply to your text column before BERTopic fitting
comment_data["comment_body"] = comment_data['comment_body'].apply(normalize_unicode).copy()

In [11]:

# Converts a raw alias dictionary into a lowercase lookup map and compiled regex pattern
def compile_entity_matcher(
    raw_map: Dict[str, List[str]]
) -> Tuple[Dict[str, List[str]], re.Pattern]:
    
    lookup_map = {}
    for canonical, aliases in raw_map.items():
        canonical_list = [canonical] if isinstance(canonical, str) else list(canonical)
        for alias in aliases:
            lookup_map[alias.lower()] = canonical_list

    sorted_aliases = sorted(lookup_map.keys(), key=len, reverse=True)
    pattern = r"\b(" + "|".join(re.escape(a) for a in sorted_aliases) + r")\b"
    return lookup_map, re.compile(pattern, flags=re.IGNORECASE)


# Precompile Regex
AEON_MAP, AEON_REGEX = compile_entity_matcher(RAW_AEONS)
NPC_MAP, NPC_REGEX = compile_entity_matcher(RAW_NPCS)
CHAR_MAP, CHAR_REGEX = compile_entity_matcher(RAW_CHARACTERS)


# Preprocessing Functions
def extract_names(text: str, lookup_map: dict, regex: re.Pattern) -> List[str]:
    """Extracts deduplicated canonical names mentioned in text."""
    if not isinstance(text, str):
        return []
    matches = regex.findall(text)
    found = set()
    for match in matches:
        found.update(lookup_map[match.lower()])
    return sorted(list(found))


def process_comment_names(text: str) -> dict:
    """Processes a single text comment to extract metadata and return masked text."""
    if not isinstance(text, str):
        return {
            "masked_text": "",
            "characters_mentioned": [],
            "npcs_mentioned": [],
            "aeons_mentioned": [],
        }

    # Extract metadata from raw text
    aeons = extract_names(text, AEON_MAP, AEON_REGEX)
    npcs = extract_names(text, NPC_MAP, NPC_REGEX)
    chars = extract_names(text, CHAR_MAP, CHAR_REGEX)

    # Mask in sequential order (Aeons -> NPCs -> Characters)
    masked = AEON_REGEX.sub("[AEON]", text)
    masked = NPC_REGEX.sub("[NPC]", masked)
    masked = CHAR_REGEX.sub("[CHARACTER]", masked)

    return {
        "masked_text": masked,
        "characters_mentioned": chars,
        "npcs_mentioned": npcs,
        "aeons_mentioned": aeons,
    }

processed = [process_comment_names(text) for text in comment_data["comment_body"]]
processed_df = pd.DataFrame(processed)

# Merge back into original DataFrame
comment_data = pd.concat([comment_data, processed_df], axis=1)

In [12]:
comment_data=comment_data.drop_duplicates(subset=["masked_text"]).reset_index(drop=True)

In [13]:
dataset = Dataset.from_pandas(comment_data)

In [15]:
comments = comment_data["masked_text"].astype(str).tolist()  # 500k list of text strings

In [16]:
topic_model = BERTopic.load("hsr_qwentopic_8",embedding_model=embedding_model)

In [19]:
embedding_model.max_seq_length = 256

In [21]:
### Batch comments

embeddings = embedding_model.encode(
    comments,
    batch_size=16,  
    show_progress_bar=True,
    convert_to_numpy=True,
)

Batches:   0%|          | 0/15423 [00:00<?, ?it/s]

In [ ]:
predicted_topics, predicted_probs = topic_model.transform(
    comments, embeddings=embeddings
)

In [ ]:

exact_labels = {
    -1: "Community & Fluff",
    0: "Endgame Content",
    1: "Monetization & Banners",
    2: "Main Story & Lore",
    3: "Character Design & Kits",
    4: "Meta & Powercreep",
    5: "Events & Collabs",
    6: "Bugs & Technical",
}

output_df = pd.DataFrame(
    {
        "comment_id": comment_data["comment_id"],
        "post_id": comment_data["post_id"],
        "masked_text": comments,
        "topic_id": predicted_topics,
        "topic_label": [
            exact_labels.get(t, "Community & Fluff") for t in predicted_topics
        ],
        "character_mention": comment_data["characters_mentioned"],
        "npc_mention": comment_data["npcs_mentioned"],
        "aeon_mention": comment_data["aeons_mentioned"],
        "time_analyzed": datetime.now(),
    }
)

In [ ]:

comments_df = output_df[
    [
        "comment_id",
        "post_id",
        "masked_text",
        "topic_id",
        "topic_label",
        "time_analyzed",
    ]
]


In [ ]:

mention_frames = []
mention_mappings = [
    ("character_mention", "character"),
    ("npc_mention", "npc"),
    ("aeon_mention", "aeon"),
]

for col, mention_type in mention_mappings:
    exploded = output_df[["comment_id", col]].explode(col).dropna()
    exploded = exploded[exploded[col] != ""]  # Filter out blank entries
    exploded = exploded.rename(columns={col: "entity_name"})
    exploded["mention_type"] = mention_type
    mention_frames.append(exploded)

mentions_df = pd.concat(mention_frames, ignore_index=True)


In [54]:
comments_df

,comment_id,post_id,masked_text,topic_id,topic_label,time_analyzed
0,t1_ofjecmf,1siaoou,[CHARACTER] lvl 1,4,Meta & Powercreep,2026-09-04 00:14:43.351867
1,t1_oji71j6,1t1iuvm,MC is a 5*,5,Events & Collabs,2026-09-04 00:14:43.351867
2,t1_o9kav0r,1rp58s5,Y r u gae?,5,Events & Collabs,2026-09-04 00:14:43.351867
3,t1_o8gdtw2,1rjnvn8,Yes I do,5,Events & Collabs,2026-09-04 00:14:43.351867
4,t1_ohejbjz,1srg17t,Me & who ?,5,Events & Collabs,2026-09-04 00:14:43.351867
...,...,...,...,...,...,...
246758,t1_ov6m0a2,1ul97d3,Everyone getting multiplier increases just off...,4,Meta & Powercreep,2026-09-04 00:14:43.351867
246759,t1_p7a63yu,1w4rr01,As long as the characters themselves are what ...,0,Endgame Content,2026-09-04 00:14:43.351867
246760,t1_o69d378,1r8sh61,My favourite part is that they're further deve...,0,Endgame Content,2026-09-04 00:14:43.351867
246761,t1_ot0appo,1uc0a7h,I LOVE [CHARACTER] I LOVE [CHARACTER] I LOVE [...,3,Character Design & Kits,2026-09-04 00:14:43.351867


In [ ]:

comments_df.to_sql("comment_topics", conn1, if_exists="append", index=False)
mentions_df.to_sql("comment_mentions", conn1, if_exists="append", index=False)



183830

In [66]:
conn1.commit()
conn1.close()

In [62]:
conn1.executescript("""
CREATE TABLE IF NOT EXISTS comment_topics (
    comment_id TEXT PRIMARY KEY,
    post_id TEXT,
    masked_text TEXT,
    topic_id INTEGER,
    topic_label TEXT,
    time_analyzed TIMESTAMP,
    FOREIGN KEY (comment_id) REFERENCES comments(comment_id) ON DELETE CASCADE
);

CREATE TABLE IF NOT EXISTS comment_mentions (
    mention_id INTEGER PRIMARY KEY AUTOINCREMENT,
    comment_id TEXT,
    mention_type TEXT,
    entity_name TEXT,
    FOREIGN KEY (comment_id) REFERENCES comments(comment_id)
);

""")

In [ ]:
### BELOW IS CODE FROM THE FIRST RUN

In [ ]:
import os
os.environ["HF_TOKEN"] = "HFAKcgGJIV6WvjM8UgnrQcCFyLrw3VI"

### First model load
#embedding_model = SentenceTransformer("BAAI/bge-large-en-v1.5", device=0, trust_remote_code=True)

### Models
#Qwen/Qwen3-Embedding-0.6B
#BAAI/bge-large-en-v1.5

### Save Qwen model locally
#embedding_model.save_pretrained("local_baai_hsr")

In [ ]:
embedding_model = SentenceTransformer("local_qwen_hsr", device="cuda")

In [ ]:
### GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using compute device: {device}")
if device == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")


In [ ]:

# 1. Define explicit target strings (mix formal names + Reddit slang)
zeroshot_topics = [
    "Memory of Chaos, MoC, MoC 12, 0-cycle, floor 12 clear, Pure Fiction, PF, Apocalyptic Shadow, AS, Anomaly Arbitration, AA, Knight, King Divergent Universe, Simulated Universe, DU, SU, Protocol 8, Protocol 10",
    "Monetization, banner, pulls, pity, 50/50, 75/25, stellar jades, loss, greedy",
    "Main Story, Penacony, Amphoreus, Belobog, Planarcadia, Astropolis, quest, lore, mission, cutscene, dialog, npc, aeon",
    "Character design, skin, outfit, animation, splash art, drip marketing, kit, va, voice",
    "Character powercreep, buff, novaflare, bricked, viable, tier, tier list, T0, T5, meta",  
    "Events, limited time, collabs, livestreams",
    "Login bugs, game errors, game fixes, mobile, ps5, pc, android, apple",
]



In [ ]:



# Dimensionality Reduction 
umap_model = UMAP(
    n_neighbors=30, 
    n_components=5,
    metric="cosine",
    random_state=42,
)

# HDBSCAN Density Clustering 
hdbscan_model = HDBSCAN(
    min_cluster_size=80,  # Reduced from 200 so specific endgame modes don't dump into -1
    min_samples=12,  # Prevents excessive noise within valid sub-clusters
    cluster_selection_method="eom",
    prediction_data=True,  # Required for fast weekly .transform() inference
)

# Vectorizer 
vectorizer_model = CountVectorizer(
    token_pattern=r"(?u)\b\w+\b",  # Retains 2-3 letter acronyms (MoC, PF, AS, AA)
    stop_words="english",
    ngram_range=(1, 2),  # captures "0 cycle", "drip marketing", "power creep"
    min_df=3,  # Drops one-off typos and unique spam tokens
)

ctfidf_model = ClassTfidfTransformer(
    reduce_frequent_words=True  # Penalizes omnipresent filler words ("character", "game", "just")
)

# Representation Model (Keyword Deduplication)
representation_model = {
    "KeyBERT": KeyBERTInspired(),
    "MMR": MaximalMarginalRelevance(diversity=0.35),  # Prevents redundant keyword list outputs
}

# 6. Master Zero-Shot Pipeline
topic_model = BERTopic(
    zeroshot_topic_list=zeroshot_topics,
    zeroshot_min_similarity=0.50,  
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    vectorizer_model=vectorizer_model,
    ctfidf_model=ctfidf_model,
    representation_model=representation_model,
    embedding_model=embedding_model,
    verbose=True,
)

In [ ]:
topics, _ = topic_model.fit_transform(comments, embeddings)

In [ ]:
# Extract the grouped lists for BERTopic
# Dictionary mapping for reference, labeling, or custom pipelines
core_topic_mapping = {
    "Meta and Powercreep": [4, 18, 35, 38, 45, 69, 74, 76, 80, 109, 110, 117, 119, 126, 127, 128, 129, 158],
    "Character Design": [3, 14, 39, 59, 87, 92, 93, 105, 159, 160, 161, 162],
    "Event and Collabs": [5, 31, 89, 106, 108, 115, 135, 136],
    "Main Story and Lore": [2, 19, 20, 23, 24, 30, 34, 37, 41, 43, 44, 46, 48, 56, 57, 58, 66, 77, 84, 85, 94, 95, 96, 100, 111, 113, 118, 122, 123, 124, 125, 130, 131, 138, 139, 141, 142, 143, 144, 146, 148, 150, 151, 152, 153, 154, 155],
    "Endgame Content": [0, 47, 62, 71, 86],
    "Monetization and Gacha": [1, 25, 27, 53, 68, 79, 81, 120, 121, 133],
    "Bugs and Tech": [6, 28, 50, 78, 97, 103, 114, 116],
    "Community Fluff": [-1, 7, 8, 9, 10, 11, 12, 13, 15, 16, 17, 21, 22, 26, 29, 32, 33, 36, 40, 42, 49, 51, 52, 54, 55, 60, 61, 63, 64, 65, 67, 70, 72, 73, 75, 82, 83, 88, 90, 91, 98, 99, 101, 102, 104, 107, 112, 132, 134, 137, 140, 145, 147, 149, 156, 157, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173]
}

topics_to_merge = list(core_topic_mapping.values())

# Execute the merge (assuming 'docs' is your original list of documents)
topic_model.merge_topics(comments, topics_to_merge)

clean_labels = {
    -1: "Community & Fluff",
    0: "Endgame Content",
    1: "Monetization & Banners",
    2: "Main Story & Lore",
    3: "Character Design & Kits",
    4: "Meta & Powercreep",
    5: "Events & Collabs",
    6: "Bugs & Technical",
}

topic_model.set_topic_labels(clean_labels)

topic_model.save("hsr_qwentopic_8", serialization="safetensors")

print("Successfully merged down to 8 categories and saved the model.")

#topic_model.custom_labels_ = None

In [ ]:
pd.set_option('display.max_rows', None)
pd.set_option('display.width', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)